In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "data" / "HumidityLookup.csv").exists():
    DATA_DIR = _cwd / "data"
elif (_cwd.parent / "data" / "HumidityLookup.csv").exists():
    DATA_DIR = _cwd.parent / "data"
else:
    DATA_DIR = _cwd.parent / "data"

sns.set_theme(style="whitegrid")


In [ ]:
df_Hum = pd.read_csv(DATA_DIR / "HumidityLookup.csv")
df_Ph = pd.read_csv(DATA_DIR / "PHRangeLookup.csv")
df_Soil = pd.read_csv(DATA_DIR / "SoilTextureLookup.csv")
df_Var = pd.read_csv(DATA_DIR / "PlantVariety.csv")
df_PlantType = pd.read_csv(DATA_DIR / "PlantTypeLookup.csv")
df_Hard = pd.read_csv(DATA_DIR / "PlantHardinessZoneLookup.csv")
df_Saline = pd.read_csv(DATA_DIR / "SalinityLookup.csv")
df_OrgMat = pd.read_csv(DATA_DIR / "OrganicMatterLookup.csv")
df_plant = pd.read_csv(DATA_DIR / "Plant.csv")


In [ ]:
# Unify hardiness zones: aggregate temperature ranges by numeric zone
df_Hard = df_Hard.copy()
df_Hard["UnifiedZone"] = df_Hard["Zone"].astype(str).str.extract(r"(\d+)")
merged_zone_data = (
    df_Hard.dropna(subset=["UnifiedZone"])
    .groupby("UnifiedZone", as_index=False)
    .agg({"TemperatureStartRange": "min", "TemperatureEndRange": "max"})
)
merged_zone_data["UnifiedZone"] = merged_zone_data["UnifiedZone"].astype(float)
df_Hard_new = merged_zone_data


In [ ]:
# Join the dataframes
result = (
    df_plant
    # Join PH Range Lookup
    .merge(df_Ph[['PHRangeID', 'PHRange', 'SoilType']], on='PHRangeID', how='left')
    # Join Unified Plant Hardiness Zones Lookup (matching ZoneID with UnifiedZone)
    .merge(df_Hard_new[['UnifiedZone', 'TemperatureStartRange', 'TemperatureEndRange']], 
           left_on='ZoneID', right_on='UnifiedZone', how='left')
    # Join Plant Type Lookup
    .merge(df_PlantType[['PlantTypeID', 'PlantType']], on='PlantTypeID', how='left')
    # Join Soil Texture Lookup
    .merge(df_Soil[['SoilTextureID', 'SoilTexture']], on='SoilTextureID', how='left')
    # Join Humidity Lookup
    .merge(df_Hum[['HumidityID', 'Classification']], on='HumidityID', how='left')
    # Join Organic Matter Lookup
    .merge(df_OrgMat[['OrganicMatterID', 'OrganicMatterContent']], on='OrganicMatterID', how='left')
    # Join Salinity Lookup
    .merge(df_Saline[['SalinityLevelID', 'SalinityLevel', 'Classification']], 
           on='SalinityLevelID', how='left', suffixes=('', '_Salinity'))
    # Join Plant Variety
    .merge(df_Var[['PlantID', 'PlantVarietyName']], on='PlantID', how='left')
)

# Drop unnecessary columns
result = result.drop(columns=['PlantDescription'])

# Display the result
print(result.head())

In [ ]:
# Optional: export joined table
# result.to_csv(DATA_DIR / "Joined_Data.csv", index=False)


In [ ]:
final_df = result

In [ ]:
final_df.describe()

In [ ]:
final_df.isnull().sum()

In [ ]:
# Drop rows with any missing values in the DataFrame
cleaned_df = final_df.dropna()

# Display the number of rows before and after cleaning
print("Rows before cleaning:", final_df.shape[0])
print("Rows after cleaning:", cleaned_df.shape[0])


In [ ]:
cleaned_df.info()

In [ ]:
# Optional: export cleaned table
# cleaned_df.to_csv(DATA_DIR / "cleaned_df.csv", index=False)


In [ ]:

cleaned_df['OrganicMatterContent'] = cleaned_df['OrganicMatterContent'].replace({
    'Moderate (2% - 4%)': 'Moderate',
    'Low (1% - 2%)': 'Low',
    'High (4% - 6%)': 'High'
})
#Print the updated DataFrame
cleaned_df

In [ ]:
cleaned_df['PlantType'].value_counts()

In [ ]:
# Create a copy of the cleaned_df
cleaned_df_copy4 = cleaned_df.copy()

# Check the copy to confirm it matches the original
cleaned_df_copy4.head()


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder object
label_encoder = LabelEncoder()

# Convert the 'PlantType' column to numeric labels
cleaned_df_copy4['PlantType'] = label_encoder.fit_transform(cleaned_df_copy4['PlantType'])

# Check the result
cleaned_df_copy4.head()


In [ ]:
# Get the mapping of labels to original categories
label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

# Print the mapping
print(label_mapping)


In [ ]:
cleaned_df_copy4['PHRange']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define features (X) and target (y)
X = cleaned_df_copy4.drop(columns=['PlantVarietyName','PlantID', 'SoilTextureID', 'PHRangeID', 'OrganicMatterID', 'SalinityLevelID', 'ZoneID', 'HumidityID', 'PlantTypeID', 'PlantType','PlantName','PHRange'])
y = cleaned_df_copy4['PlantType']

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True)

# Ensure all column names are valid strings
X_encoded.columns = X_encoded.columns.str.replace(r'[^\w\s]', '', regex=True)

# Split the data into training and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Train the Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {accuracy_rf:.2f}")
print("\nClassification Report for Random Forest:\n")
print(classification_report(y_test, y_pred_rf))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming `rf_model` is your trained Random Forest model
feature_importances = rf_model.feature_importances_

# Sort feature importances in descending order
sorted_idx = np.argsort(feature_importances)[::-1]

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importances)), feature_importances[sorted_idx], align='center')
plt.yticks(range(len(feature_importances)), np.array(X_encoded.columns)[sorted_idx])
plt.gca().invert_yaxis()  # Invert y-axis to display top features at the top
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.show()


In [ ]:
cleaned_df.PlantType.value_counts()

In [ ]:
sns.catplot(
    data=cleaned_df, 
    x="SoilType", 
    hue="PlantType", 
    kind="count", 
    height=6, 
    aspect=2, 
    palette="Set2"
)
plt.title('Soil Type Suitability for Plant Types')
plt.ylabel('Count')
plt.xlabel('Soil Type')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.catplot(
    x='UnifiedZone', 
    hue='PlantType', 
    kind='count', 
    data=cleaned_df, 
    height=6, 
    aspect=2, 
    palette='muted'
)
plt.title('Distribution of Plant Types Across Zones')
plt.ylabel('Count')
plt.xlabel('Unified Zone')
plt.xticks(rotation=45)
plt.show()